# Feature Engineering

Feature engineering is the process of creating new variables from existing data to improve the predictive performance of machine learning models.

The engineered features are based on domain knowledge of customer behavior in the telecommunications industry.

In [27]:
import pandas as pd
import numpy as np

In [28]:
df = pd.read_csv("../datasets/processed/cleaned_telco.csv")

In [29]:
df["AvgMonthlySpend"] = np.where(
    df["tenure"] == 0,
    0,
    df["TotalCharges"] / df["tenure"]
)

df[["MonthlyCharges", "AvgMonthlySpend"]].head()

,MonthlyCharges,AvgMonthlySpend
0,29.85,29.850000
1,56.95,55.573529
2,53.85,54.075000
3,42.30,40.905556
4,70.70,75.825000


Average Monthly Spend represents the customer's average expenditure over their subscription period.

Customers with unusually high average spending may have different churn behavior compared to low-spending customers.

In [30]:
service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["NumServices"] = (
    df[service_columns]
    .isin(["Yes"])
    .sum(axis=1)
)

df["NumServices"].head()

0    1
1    3
2    3
3    3
4    1
Name: NumServices, dtype: int64

NumServices measures customer engagement by counting the number of subscribed services.

Customers using multiple services are generally less likely to leave because switching becomes more costly.

In [31]:
def loyalty_group(x):
    if x <= 12:
        return "New"
    elif x <= 48:
        return "Regular"
    else:
        return "Loyal"


df["CustomerLoyalty"] = df["tenure"].apply(loyalty_group)

df["CustomerLoyalty"].value_counts()

CustomerLoyalty
Regular    2618
Loyal      2239
New        2186
Name: count, dtype: int64

Customers are divided into loyalty groups based on tenure.

- New (0–12 months)
- Regular (13–48 months)
- Loyal (>48 months)

Long-term customers are generally less likely to churn.

In [32]:
threshold = df["MonthlyCharges"].median()

df["HighMonthlyCharge"] = (
    df["MonthlyCharges"] > threshold
).astype(int)

df["HighMonthlyCharge"].value_counts()

HighMonthlyCharge
0    3528
1    3515
Name: count, dtype: int64

Customers paying above the median monthly charge are marked as high-paying customers.

This binary feature may improve model performance by simplifying spending behavior.

In [33]:
df["HighValueCustomer"] = np.where(
    (df["MonthlyCharges"] > df["MonthlyCharges"].median()) &
    (df["tenure"] > 24),
    1,
    0
)

df["HighValueCustomer"].head()

0    0
1    0
2    0
3    0
4    0
Name: HighValueCustomer, dtype: int64

A High Value Customer has both

- high monthly spending
- long subscription history

These customers are especially important for retention strategies.

In [34]:
auto_methods = [
    "Bank transfer (automatic)",
    "Credit card (automatic)"
]

df["AutoPayment"] = (
    df["PaymentMethod"]
    .isin(auto_methods)
    .astype(int)
)

df["AutoPayment"].head()

0    0
1    0
2    0
3    1
4    0
Name: AutoPayment, dtype: int64

Customers using automatic payment methods often demonstrate stronger commitment and lower churn rates.

In [35]:
df["PaperlessAutoPay"] = np.where(
    (df["PaperlessBilling"] == "Yes") &
    (df["AutoPayment"] == 1),
    1,
    0
)

df["PaperlessAutoPay"].head()

0    0
1    0
2    0
3    0
4    0
Name: PaperlessAutoPay, dtype: int64

This feature identifies customers who use both paperless billing and automatic payments, representing digitally engaged customers.

In [36]:
df.drop(columns="customerID", inplace=True)

CustomerID is unique for every customer and does not contain predictive information, so it is removed before model training.

In [37]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,NumServices,CustomerLoyalty,HighMonthlyCharge,HighValueCustomer,AutoPayment,PaperlessAutoPay
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,29.85,29.85,No,29.850000,1,New,0,0,0,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,56.95,1889.50,No,55.573529,3,Regular,0,0,0,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,53.85,108.15,Yes,54.075000,3,New,0,0,0,0
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,42.30,1840.75,No,40.905556,3,Regular,0,0,1,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,70.70,151.65,Yes,75.825000,1,New,1,0,0,0


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             7043 non-null   object 
 1   SeniorCitizen      7043 non-null   int64  
 2   Partner            7043 non-null   object 
 3   Dependents         7043 non-null   object 
 4   tenure             7043 non-null   int64  
 5   PhoneService       7043 non-null   object 
 6   MultipleLines      7043 non-null   object 
 7   InternetService    7043 non-null   object 
 8   OnlineSecurity     7043 non-null   object 
 9   OnlineBackup       7043 non-null   object 
 10  DeviceProtection   7043 non-null   object 
 11  TechSupport        7043 non-null   object 
 12  StreamingTV        7043 non-null   object 
 13  StreamingMovies    7043 non-null   object 
 14  Contract           7043 non-null   object 
 15  PaperlessBilling   7043 non-null   object 
 16  PaymentMethod      7043 

In [39]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn', 'AvgMonthlySpend',
       'NumServices', 'CustomerLoyalty', 'HighMonthlyCharge',
       'HighValueCustomer', 'AutoPayment', 'PaperlessAutoPay'],
      dtype='object')

## Feature Engineering Summary

The following features were created:

- AvgMonthlySpend
- NumServices
- CustomerLoyalty
- HighMonthlyCharge
- HighValueCustomer
- AutoPayment
- PaperlessAutoPay

The CustomerID column was removed because it has no predictive value.

These engineered features capture customer engagement, loyalty, payment behavior, and spending patterns, which are expected to improve churn prediction performance.

In [40]:
df.to_csv("../datasets/processed/cleaned_telco2.csv", index=False)